In [10]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
import chromadb
from pathlib import Path
import json
from langchain_community.document_loaders import JSONLoader
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"
#Chunk folders
recursive_1000 = DATA_DIR / "chunks" / "recursive_1000"
rsc = DATA_DIR / "chunks" / "rsc"
semantic_p95 = DATA_DIR / "chunks" / "semantic_p95"

In [ ]:
#Database
chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
colRecursive = chroma_client.get_or_create_collection(name="recursive_100")#was meant to be 1000, messed up up
colSemantic = chroma_client.get_or_create_collection(name="semantic_p95")
colRsc = chroma_client.get_or_create_collection(name="rsc")

In [13]:
options = [
    "BAAI/bge-base-en-v1.5",#440MB, 768 dimensions. What Medium tutorial uses. Solid retrieval performance, a step up from MiniLM in quality, still runs fine on CPU.
    "BAAI/bge-m3"# 2.2GB, 1024 dimensions. Top multilingual model with a score of 63.0 on MTEB, supporting 100+ languages.Relevant because corpus has Korean, Latin species names, and English mixed together.
]

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1557.27it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Recursive 1000 Chunking 


In [16]:
all_ids = []
all_texts = []
all_metadatas = []

for json_path in sorted(recursive_1000.glob("*.json")):
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)
    
    for chunk in doc["chunks"]:
        all_ids.append(f"{doc['filename']}_chunk_{chunk['chunk_id']:03d}")
        all_texts.append(chunk["text"])
        all_metadatas.append({
            "filename": doc["filename"],
            "title": doc.get("title") or "",
            "authors": ", ".join(doc.get("authors") or []),
            "year": doc.get("year") or "",
        })

print(f"Loaded {len(all_ids)} chunks")


Loaded 31741 chunks


In [15]:
BATCH_SIZE = 50

for start in range(0, len(all_ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(all_ids))
    
    embeddings = embedding_model.embed_documents(all_texts[start:end])
    
    colRecursive.add(
        ids=all_ids[start:end],
        embeddings=embeddings,
        documents=all_texts[start:end],
        metadatas=all_metadatas[start:end],
    )
    
    if (start + BATCH_SIZE) % 500 == 0:
        print(f"[{end}/{len(all_ids)}] embedded")

print(f"Done. {colRecursive.count()} chunks in collection.")

[500/31741] embedded
[1000/31741] embedded
[1500/31741] embedded
[2000/31741] embedded
[2500/31741] embedded
[3000/31741] embedded
[3500/31741] embedded
[4000/31741] embedded
[4500/31741] embedded
[5000/31741] embedded
[5500/31741] embedded
[6000/31741] embedded
[6500/31741] embedded
[7000/31741] embedded
[7500/31741] embedded
[8000/31741] embedded
[8500/31741] embedded
[9000/31741] embedded
[9500/31741] embedded
[10000/31741] embedded
[10500/31741] embedded
[11000/31741] embedded
[11500/31741] embedded
[12000/31741] embedded
[12500/31741] embedded
[13000/31741] embedded
[13500/31741] embedded
[14000/31741] embedded
[14500/31741] embedded
[15000/31741] embedded
[15500/31741] embedded
[16000/31741] embedded
[16500/31741] embedded
[17000/31741] embedded
[17500/31741] embedded
[18000/31741] embedded
[18500/31741] embedded
[19000/31741] embedded
[19500/31741] embedded
[20000/31741] embedded
[20500/31741] embedded
[21000/31741] embedded
[21500/31741] embedded
[22000/31741] embedded
[22500/3

## Semantic Chunking

In [17]:
all_ids = []
all_texts = []
all_metadatas = []

for json_path in sorted(semantic_p95.glob("*.json")):
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)
    
    for chunk in doc["chunks"]:
        all_ids.append(f"{doc['filename']}_chunk_{chunk['chunk_id']:03d}")
        all_texts.append(chunk["text"])
        all_metadatas.append({
            "filename": doc["filename"],
            "title": doc.get("title") or "",
            "authors": ", ".join(doc.get("authors") or []),
            "year": doc.get("year") or "",
        })

print(f"Loaded {len(all_ids)} chunks")


Loaded 14741 chunks


In [18]:
BATCH_SIZE = 50

for start in range(0, len(all_ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(all_ids))
    
    embeddings = embedding_model.embed_documents(all_texts[start:end])
    
    colSemantic.add(
        ids=all_ids[start:end],
        embeddings=embeddings,
        documents=all_texts[start:end],
        metadatas=all_metadatas[start:end],
    )
    
    if (start + BATCH_SIZE) % 500 == 0:
        print(f"[{end}/{len(all_ids)}] embedded")

print(f"Done. {colSemantic.count()} chunks in collection.")

[500/14741] embedded
[1000/14741] embedded
[1500/14741] embedded
[2000/14741] embedded
[2500/14741] embedded
[3000/14741] embedded
[3500/14741] embedded
[4000/14741] embedded
[4500/14741] embedded
[5000/14741] embedded
[5500/14741] embedded
[6000/14741] embedded
[6500/14741] embedded
[7000/14741] embedded
[7500/14741] embedded
[8000/14741] embedded
[8500/14741] embedded
[9000/14741] embedded
[9500/14741] embedded
[10000/14741] embedded
[10500/14741] embedded
[11000/14741] embedded
[11500/14741] embedded
[12000/14741] embedded
[12500/14741] embedded
[13000/14741] embedded
[13500/14741] embedded
[14000/14741] embedded
[14500/14741] embedded
Done. 14741 chunks in collection.


## Recursive-Semantic Chunking

In [19]:
all_ids = []
all_texts = []
all_metadatas = []

for json_path in sorted(rsc.glob("*.json")):
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)
    
    for chunk in doc["chunks"]:
        all_ids.append(f"{doc['filename']}_chunk_{chunk['chunk_id']:03d}")
        all_texts.append(chunk["text"])
        all_metadatas.append({
            "filename": doc["filename"],
            "title": doc.get("title") or "",
            "authors": ", ".join(doc.get("authors") or []),
            "year": doc.get("year") or "",
        })

print(f"Loaded {len(all_ids)} chunks")


Loaded 25524 chunks


In [1]:
BATCH_SIZE = 50

for start in range(0, len(all_ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(all_ids))
    
    embeddings = embedding_model.embed_documents(all_texts[start:end])
    
    colRsc.add(
        ids=all_ids[start:end],
        embeddings=embeddings,
        documents=all_texts[start:end],
        metadatas=all_metadatas[start:end],
    )
    
    if (start + BATCH_SIZE) % 500 == 0:
        print(f"[{end}/{len(all_ids)}] embedded")

print(f"Done. {colRsc.count()} chunks in collection.")

NameError: name 'all_ids' is not defined